# Lewisham · `url_planning_app` (web: "Planning application URL")

**Goal:** understand the data, so that a chat bot can answer any question about it
more easily and correctly. Lewisham also serves as a check on what we found for Brent.

Lewisham only (golden rule): every query carries the location-based Lewisham filter.
Data comes from the **live API** (the bot's only source), so counts change as the
Datahub is updated.

Workflow: add a step here → discuss the findings → only after agreement update
`semantic/lewisham/` and `knowledge/`.

In [1]:
from pathlib import Path

import httpx
import pandas as pd
from dotenv import dotenv_values

PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
env = dotenv_values(PROJECT_ROOT / ".env")

BASE_URL = "https://planningdata.london.gov.uk/api-guest"
HEADERS = {"X-API-AllowRequest": env["DATAHUB_ALLOW_REQUEST"]}


def lewisham_filter() -> dict:
    # Location-based scope: borough is Lewisham, or borough is missing and Lewisham decided it.
    return {
        "bool": {
            "should": [
                {"term": {"borough.raw": "Lewisham"}},
                {"bool": {
                    "filter": [{"term": {"lpa_name.raw": "Lewisham"}}],
                    "must_not": [{"exists": {"field": "borough.raw"}}],
                }},
            ],
            "minimum_should_match": 1,
        }
    }


def search(body: dict) -> dict:
    response = httpx.post(f"{BASE_URL}/applications/_search", headers=HEADERS, json=body, timeout=60)
    response.raise_for_status()
    return response.json()

## Step 1 · Nulls and empty values

Fetch `url_planning_app` for every Lewisham record (paged with `search_after`, sorted by
`id`), with `bo_system` and the four fields of the empty-record rule, so legacy and empty
records can be shown apart. Count null, empty or blank, and filled.

In [2]:
COLS = ["id", "url_planning_app", "bo_system", "status", "decision", "decision_date", "description"]
rows, after = [], None
while True:
    body = {"size": 5000, "query": lewisham_filter(), "_source": COLS, "sort": [{"id": "asc"}]}
    if after:
        body["search_after"] = after
    page = search(body)["hits"]["hits"]
    if not page:
        break
    rows += [h["_source"] for h in page]
    after = page[-1]["sort"]

urls = pd.DataFrame(rows).reindex(columns=COLS)


def is_empty(series: pd.Series) -> pd.Series:
    return series.isna() | (series.astype(str).str.strip() == "")


urls["empty_record"] = pd.concat([is_empty(urls[f]) for f in ["status", "decision", "decision_date", "description"]], axis=1).all(axis=1)
urls["legacy"] = urls["bo_system"] == "FME Legacy Data Import"
v = urls["url_planning_app"]
is_blank = v.notna() & (v.astype(str).str.strip() == "")
urls["no_url"] = v.isna() | is_blank
print("Lewisham records:", len(urls))
print({"null": int(v.isna().sum()), "empty or blank": int(is_blank.sum()), "filled": int((~urls["no_url"]).sum())})
print()
print("No URL, by group:")
group = urls["legacy"].map({True: "legacy", False: ""}) + urls["empty_record"].map({True: "empty", False: ""})
print(pd.crosstab(group.replace("", "other").rename("group"), urls["no_url"].rename("no URL")).to_string())
print()
print("No URL, other records, by bo_system:", urls.loc[urls["no_url"] & ~urls["legacy"] & ~urls["empty_record"], "bo_system"].value_counts().to_dict())

Lewisham records: 29021
{'null': 5528, 'empty or blank': 0, 'filled': 23493}

No URL, by group:
no URL  False  True 
group               
empty       1   1150
legacy      0   4312
other   23492     66

No URL, other records, by bo_system: {'CSV Data Import': 36, 'Portal': 30}


### Findings · Step 1

Live API, 2026-11-11, 29,021 Lewisham records:

| | Count |
|---|---|
| null | 5,528 |
| empty or blank | 0 |
| filled | 23,493 |

| group | no URL | URL |
|---|---|---|
| legacy (FME Legacy Data Import) | **4,312** | 0 |
| empty records (dropped) | 1,150 | 1 |
| other | **66** | 23,492 |

- **Legacy records never have a URL** (all 4,312), as in Brent.
- **66 other records have no URL:** 36 from the CSV import and 30 of the 33 `Portal`
  records. These include the 49 with no valid_date (valid_date notebook, Step 3); the
  rest are not listed yet. Brent had 97 such real gaps.

## Step 2 · URL format, uniqueness, and the 17 unexplained gaps

1. Every filled URL against the expected pattern
   `https://planning.lewisham.gov.uk/online-applications/applicationDetails.do?keyVal=<key>&activeTab=summary`
   (exact host and path, one key); anything else listed. Then whether two records share a
   URL.
2. The 66 "other" records without a URL, minus the 49 that also have no valid_date
   (valid_date notebook, Step 3): the rest, listed.

In [3]:
import re

PATTERN = re.compile(r"^https://planning\.lewisham\.gov\.uk/online-applications/applicationDetails\.do\?keyVal=([A-Za-z0-9_]+)&activeTab=summary$")
filled = urls.loc[~urls["no_url"], ["id", "url_planning_app"]].copy()
filled["key"] = filled["url_planning_app"].str.extract(PATTERN, expand=False)
print("Filled:", len(filled), "| match the pattern:", int(filled["key"].notna().sum()))
print("Not matching:", filled.loc[filled["key"].isna(), "url_planning_app"].head(10).tolist())
print("Key prefixes:", filled["key"].str.extract(r"^([A-Za-z_]+)", expand=False).value_counts().to_dict())
dup = filled[filled.duplicated("url_planning_app", keep=False)].sort_values("url_planning_app")
print("Records sharing a URL:", len(dup))
print(dup.head(20).to_string(index=False))

Filled: 23493 | match the pattern: 23493
Not matching: []
Key prefixes: {'_LEWIS_DCAPR_': 23493}
Records sharing a URL: 158
                   id                                                                                                        url_planning_app                 key
Lewisham-DC_20_116497 https://planning.lewisham.gov.uk/online-applications/applicationDetails.do?keyVal=_LEWIS_DCAPR_104416&activeTab=summary _LEWIS_DCAPR_104416
   Lewisham-20_116497 https://planning.lewisham.gov.uk/online-applications/applicationDetails.do?keyVal=_LEWIS_DCAPR_104416&activeTab=summary _LEWIS_DCAPR_104416
   Lewisham-21_120061 https://planning.lewisham.gov.uk/online-applications/applicationDetails.do?keyVal=_LEWIS_DCAPR_107982&activeTab=summary _LEWIS_DCAPR_107982
Lewisham-DC_21_120061 https://planning.lewisham.gov.uk/online-applications/applicationDetails.do?keyVal=_LEWIS_DCAPR_107982&activeTab=summary _LEWIS_DCAPR_107982
Lewisham-DC_21_120074 https://planning.lewisham.gov.uk/online-appl

In [4]:
# The other records without a URL that do have a valid_date.
gap = urls[urls["no_url"] & ~urls["legacy"] & ~urls["empty_record"]]
more = search({"size": 100, "_source": ["id", "valid_date", "status", "decision", "decision_date", "application_type", "bo_system", "last_updated"],
               "sort": [{"id": "asc"}],
               "query": {"bool": {"filter": [lewisham_filter(), {"terms": {"id": gap["id"].tolist()}}, {"exists": {"field": "valid_date"}}]}}})
rest = pd.DataFrame([h["_source"] for h in more["hits"]["hits"]])
rest["last_updated"] = rest["last_updated"].str.slice(0, 10)
print(len(gap), "without a URL;", len(gap) - len(rest), "of them have no valid_date;", len(rest), "have one:")
rest[["id", "bo_system", "application_type", "status", "decision", "valid_date", "decision_date", "last_updated"]]

66 without a URL; 49 of them have no valid_date; 17 have one:


,id,bo_system,application_type,status,decision,valid_date,decision_date,last_updated
0,Lewisham-DC_10_075331,CSV Data Import,All Other,Approved,Approved,14/09/2010,30/03/2012,2025-09-09
1,Lewisham-DC_14_087819,CSV Data Import,All Other,Completed,Approved,20/06/2014,07/07/2016,2025-03-19
2,Lewisham-DC_16_098591,CSV Data Import,All Other,Completed,Approved,13/09/2016,18/11/2016,2025-03-19
3,Lewisham-DC_16_099036,Portal,Prior Approval,Completed,Approved,08/11/2016,21/12/2016,2025-08-14
4,Lewisham-DC_17_099662,CSV Data Import,All Other,NaN,Approved,06/01/2017,01/05/2020,2025-08-29
5,Lewisham-DC_17_101621,Portal,All Other,Lapsed,Refused,17/05/2017,21/05/2018,2025-12-09
6,Lewisham-DC_17_103574,CSV Data Import,All Other,Completed,Approved,21/09/2017,16/11/2017,2025-03-19
7,Lewisham-DC_17_104640,CSV Data Import,All Other,Completed,Approved,20/11/2017,30/04/2020,2025-03-20
8,Lewisham-DC_17_105015,Portal,All Other,Completed,Approved,14/12/2017,22/02/2018,2025-09-18
9,Lewisham-DC_18_106708,CSV Data Import,All Other,Completed,Approved,16/05/2018,22/05/2020,2025-03-20


In [5]:
# The shared URLs: how many groups, and how the two references differ.
dup = dup.copy()
dup["ref"] = dup["id"].str.replace("Lewisham-", "", regex=False)
groups = dup.groupby("url_planning_app")["ref"].apply(lambda s: sorted(s))
print("URLs shared:", len(groups), "| sizes:", groups.map(len).value_counts().to_dict())


def pattern(pair: list[str]) -> str:
    a, b = pair
    if a.replace("DC_", "") == b or b.replace("DC_", "") == a:
        return "same number, one without DC_"
    if a[-6:] == b[-6:]:
        return "same number, different year"
    return "other"


print(groups[groups.map(len) == 2].map(pattern).value_counts().to_dict())
print([g for g in groups if pattern(g) == "other"][:10] if len(groups) else "")

URLs shared: 79 | sizes: {2: 79}
{'same number, one without DC_': 74, 'same number, different year': 5}
[]


### Findings · Step 2

**Format:** all 23,493 URLs match the pattern exactly (one host, one path, keys
`_LEWIS_DCAPR_<n>`). No spaces or other hosts (Brent had 2 with a space).

**Not unique: 79 URLs are shared by 2 records each (158 records).**
- **74 pairs: the same reference with and without "DC/"**, e.g. `DC/20/116497` and
  `20/116497`. These are most of the 106 references without a prefix (lpa_app_no
  notebook, Step 3).
- **5 pairs: the same number with a different year**, e.g. `DC/22/125155` and
  `DC/21/125155`.
- So **Lewisham has duplicate records** that the lpa_app_no check (Step 1 there) could not
  see, because their references differ. Brent's duplicates (Brent/OPDC) were found by
  reference; these only show through the shared portal page. Which copy is right is not
  checked yet.

**The 17 other records without a URL** (they have a valid_date): all decided except
DC/18/108548 (pending since 2018, seen before); 6 Portal, 11 CSV import; decided
2012–2022, most status Completed; last_updated 2025-03 to 2025-12. No pattern beyond
"older, mostly built".

## Step 3 · The 79 pairs side by side

For each pair sharing a URL, fetch both records and compare `status`, `decision`, the
four dates, `application_type`, `bo_system`, `last_updated`, `description` and
`site_name`. Copy **A** is the record whose reference starts with "DC/" (for the 5
different-year pairs, the later year); copy **B** is the other.

In [6]:
CMP = ["status", "decision", "decision_date", "valid_date", "decision_target_date", "appeal_decision",
       "application_type", "bo_system", "last_updated", "description", "site_name", "lpa_app_no"]
ids = dup["id"].tolist()
hits = search({"size": 500, "_source": ["id", *CMP],
               "query": {"bool": {"filter": [lewisham_filter(), {"terms": {"id": ids}}]}}})["hits"]["hits"]
rec = pd.DataFrame([h["_source"] for h in hits]).set_index("id")

pairs = []
for url, refs_ in groups.items():
    a_ref, b_ref = sorted(refs_, key=lambda r: (not r.startswith("DC_"), [-ord(c) for c in r]))
    a, b = rec.loc["Lewisham-" + a_ref], rec.loc["Lewisham-" + b_ref]
    row = {"A": a["lpa_app_no"], "B": b["lpa_app_no"]}
    for f in CMP[:-1]:
        va, vb = a.get(f), b.get(f)
        same = (pd.isna(va) and pd.isna(vb)) or va == vb
        row[f] = "same" if same else ("only A" if pd.isna(vb) else ("only B" if pd.isna(va) else "differ"))
    row["A_last_updated"], row["B_last_updated"] = str(a["last_updated"])[:10], str(b["last_updated"])[:10]
    row["A_bo"], row["B_bo"] = a["bo_system"], b["bo_system"]
    pairs.append(row)
pairs = pd.DataFrame(pairs)
print(len(pairs), "pairs")
summary = pd.DataFrame({f: pairs[f].value_counts() for f in CMP[:-1]}).fillna(0).astype(int).T
summary

79 pairs


,differ,only A,only B,same
status,74,0,0,5
decision,1,35,36,7
decision_date,0,35,35,9
valid_date,1,0,0,78
decision_target_date,2,11,8,58
appeal_decision,0,3,0,76
application_type,0,0,0,79
bo_system,1,0,0,78
last_updated,79,0,0,0
description,33,0,0,46


In [7]:
print("bo_system A/B:", (pairs["A_bo"] + " / " + pairs["B_bo"]).value_counts().to_dict())
newer = pd.to_datetime(pairs["A_last_updated"]) - pd.to_datetime(pairs["B_last_updated"])
print("last_updated: A newer", int((newer.dt.days > 0).sum()), "| B newer", int((newer.dt.days < 0).sum()), "| same day", int((newer.dt.days == 0).sum()))
print()
diff = pairs[(pairs[["status", "decision", "decision_date", "valid_date"]] != "same").any(axis=1)]
print(len(diff), "pairs differ in status, decision or the main dates:")
pd.set_option("display.width", 300)
print(diff[["A", "B", "status", "decision", "decision_date", "valid_date", "A_last_updated", "B_last_updated"]].to_string(index=False))

bo_system A/B: {'CSV Data Import / CSV Data Import': 78, 'Portal / CSV Data Import': 1}
last_updated: A newer 39 | B newer 40 | same day 0

74 pairs differ in status, decision or the main dates:
           A              B status decision decision_date valid_date A_last_updated B_last_updated
DC/20/116497      20/116497 differ   differ          same       same     2026-01-20     2026-11-10
DC/21/120061      21/120061 differ   only B        only B       same     2025-05-12     2026-11-10
DC/21/120074      21/120074 differ   only B        only B       same     2025-05-12     2026-11-10
DC/21/120075      21/120075 differ   only B        only B       same     2025-05-12     2026-11-10
DC/21/120309      21/120309 differ   only B        only B       same     2025-05-12     2026-11-10
DC/21/120357      21/120357 differ   only B        only B       same     2025-05-12     2026-11-10
DC/21/122056      21/122056 differ   only A        only A       same     2026-11-10     2025-05-12
DC/21/123241 

In [8]:
# Newer vs older copy in each pair: last_updated, status, decision.
rows = []
for r in pairs.itertuples():
    a, b = rec.loc["Lewisham-" + r.A.replace("/", "_")], rec.loc["Lewisham-" + r.B.replace("/", "_")]
    new, old = (a, b) if str(a["last_updated"]) > str(b["last_updated"]) else (b, a)
    rows.append({"newer": new["lpa_app_no"], "newer_updated": str(new["last_updated"])[:10], "newer_status": new["status"],
                 "newer_decision": new["decision"], "newer_has_decision_date": pd.notna(new["decision_date"]),
                 "older": old["lpa_app_no"], "older_updated": str(old["last_updated"])[:10], "older_status": old["status"],
                 "older_decision": old["decision"], "older_has_decision_date": pd.notna(old["decision_date"])})
nv = pd.DataFrame(rows)
print("newer copy updated on:", nv["newer_updated"].value_counts().to_dict())
print("older copy updated on:", nv["older_updated"].value_counts().to_dict())
print("older copy status:", nv["older_status"].fillna("(none)").value_counts().to_dict())
print("newer copy status:", nv["newer_status"].fillna("(none)").value_counts().to_dict())
print("newer has decision_date:", int(nv["newer_has_decision_date"].sum()), "| older has:", int(nv["older_has_decision_date"].sum()))
print("older copy is the one with DC/ prefix:", int(nv["older"].str.startswith("DC/").sum()), "of", len(nv))
print()
print(nv[nv["older_has_decision_date"] | ~nv["newer_has_decision_date"]].to_string(index=False))

newer copy updated on: {'2026-11-10': 79}
older copy updated on: {'2025-05-12': 74, '2026-01-20': 1, '2026-06-24': 1, '2025-03-19': 1, '2026-10-07': 1, '2026-10-13': 1}
older copy status: {'Application Under Consideration': 71, 'Approved': 4, 'Refused': 2, 'Withdrawn': 1, 'Appeal in Progress': 1}
newer copy status: {'Approved': 53, 'Refused': 13, 'Lapsed': 5, 'Dismissed': 3, 'Completed': 2, 'Withdrawn': 2, 'Application Invalid': 1}
newer has decision_date: 78 | older has: 8
older copy is the one with DC/ prefix: 41 of 79

         newer newer_updated        newer_status newer_decision  newer_has_decision_date        older older_updated                    older_status older_decision  older_has_decision_date
     20/116497    2026-11-10            Approved       Approved                     True DC/20/116497    2026-01-20                       Withdrawn      Withdrawn                     True
  DC/22/126855    2026-11-10           Completed       Approved                     True    22/1

In [9]:
# DC/18/022222 shares its URL with the test record TEST/18/022222.
pd.DataFrame([h["_source"] for h in search({"size": 2, "_source": ["id", "status", "decision", "valid_date", "site_name", "description", "bo_system", "last_updated"],
    "query": {"bool": {"filter": [lewisham_filter(), {"terms": {"id": ["Lewisham-DC_18_022222", "Lewisham-TEST_18_022222"]}}]}}})["hits"]["hits"]]).T

,0,1
bo_system,CSV Data Import,CSV Data Import
site_name,"GAOLHOUSE COURT, 114-116 AMERSHAM VALE, LONDON...","LEE GREEN TELEPHONE EXCHANGE, LEE HIGH ROAD, L..."
last_updated,2025-05-12T10:52:42.078Z,2026-11-10T10:00:02.785Z
valid_date,18/07/2023,22/09/2026
decision,NaN,Refused
description,Application for a Lawful Development Certifica...,Application for a Lawful Development Certifica...
id,Lewisham-DC_18_022222,Lewisham-TEST_18_022222
status,Application Under Consideration,Application Invalid


### Findings · Step 3

**The pairs are the same application, recorded twice; one copy is current, one is old.**
- Same in almost every pair: valid_date (78 of 79), application_type (79), bo_system (78),
  site_name (76). description differs in 33 (wording only, not checked one by one).
- **The newer copy was updated on 2026-11-10 in all 79 pairs**; the older one mostly on
  2025-05-12 (74), so it stopped being updated.
- **The older copy is mostly still pending:** 71 have status Application Under
  Consideration, while the newer copy has the outcome (Approved 53, Refused 13, Lapsed 5,
  Dismissed 3, …). 78 newer copies have a decision_date; 8 older ones do.
- **The prefix does not tell which is current:** the older copy is the "DC/" one in 41
  pairs and the one without "DC/" in 38.
- **Effect if both are kept:** about 71 decided applications also count as pending (the 552
  pending of the status notebook include them), and 79 applications count twice.

**Two pairs need a look:**
- `DC/20/116497` (Portal, updated 2026-01-20): status and decision **Withdrawn**;
  `20/116497` (updated 2026-11-10): **Approved**. The two copies disagree on the outcome.
- `DC/18/022222` (Gaolhouse Court, 114-116 Amersham Vale; valid 18/07/2023; Application
  Under Consideration; updated 2025-05-12) shares its URL with the **test record**
  `TEST/18/022222` (updated 2026-11-10; dropped). Whether DC/18/022222 is real is not
  known.

## Step 4 · The 77 older copies to drop (for review)

The older copy of each pair (Step 3), leaving out the two pairs that need a look
(DC/20/116497 / 20/116497 and DC/18/022222 / TEST/18/022222). The newer copy, updated
2026-11-10, is kept.

In [10]:
HOLD = {"DC/20/116497", "20/116497", "DC/18/022222", "TEST/18/022222"}
drop = nv[~nv["older"].isin(HOLD) & ~nv["newer"].isin(HOLD)].copy()
drop["drop_id"] = "Lewisham-" + drop["older"].str.replace("/", "_")
print(len(drop), "ids to drop")
print("older copy status:", drop["older_status"].fillna("(none)").value_counts().to_dict())
pd.set_option("display.max_rows", 100); pd.set_option("display.width", 300)
drop[["drop_id", "older_updated", "older_status", "newer", "newer_status"]].sort_values("drop_id").reset_index(drop=True)

77 ids to drop
older copy status: {'Application Under Consideration': 70, 'Approved': 4, 'Refused': 2, 'Appeal in Progress': 1}


,drop_id,older_updated,older_status,newer,newer_status
0,Lewisham-21_122056,2025-05-12,Application Under Consideration,DC/21/122056,Approved
1,Lewisham-21_123241,2025-05-12,Application Under Consideration,DC/21/123241,Approved
2,Lewisham-22_126072,2025-05-12,Application Under Consideration,DC/22/126072,Approved
3,Lewisham-22_126814,2025-05-12,Application Under Consideration,DC/22/126814,Refused
4,Lewisham-22_126855,2025-05-12,Approved,DC/22/126855,Completed
5,Lewisham-22_127022,2025-05-12,Appeal in Progress,DC/22/127022,Dismissed
6,Lewisham-22_127239,2025-05-12,Application Under Consideration,DC/22/127239,Approved
7,Lewisham-22_127549,2025-05-12,Application Under Consideration,DC/22/127549,Approved
8,Lewisham-22_127581,2025-05-12,Application Under Consideration,DC/22/127581,Approved
9,Lewisham-22_127819,2025-05-12,Application Under Consideration,DC/22/127819,Refused


## Step 5 · `url_planning_app` after cleaning

The drops of `semantic/lewisham/cleaning.yaml` applied by hand (ids and the empty-record
condition read from the file), then: URLs filled, missing (legacy / other), and whether
every URL is now unique.

In [11]:
import yaml

rules = yaml.safe_load((PROJECT_ROOT / "semantic/lewisham/cleaning.yaml").read_text())
keep = pd.Series(True, index=urls.index)
for rule in rules["all_records"]:
    if rule["action"] == "drop_records":
        hit = urls["id"].isin(rule.get("ids", []))
        if "when_all_null" in rule:
            hit |= urls[rule["when_all_null"]].map(lambda v: not isinstance(v, str) or not v.strip()).all(axis=1)
        keep &= ~hit
after = urls[keep]
print("Records after cleaning:", len(after), "| with URL:", int((~after["no_url"]).sum()),
      "| without:", int(after["no_url"].sum()), "(legacy", int((after["no_url"] & after["legacy"]).sum()),
      "| other", int((after["no_url"] & ~after["legacy"]).sum()), ")")
print("URLs shared by two records after cleaning:", int(after.loc[~after["no_url"], "url_planning_app"].duplicated().sum()))

Records after cleaning: 27765 | with URL: 23387 | without: 4378 (legacy 4312 | other 66 )
URLs shared by two records after cleaning: 0


### Findings · Step 5

- **27,765 records after cleaning: 23,387 with a URL, 4,378 without** (legacy 4,312, other 66).
- **Every URL is unique after cleaning** (0 shared): the drops resolve all 79 shared
  pages.